# 롤 API 데이터 수집하기

In [208]:
import requests
import json
import time
import numpy as np
import pandas as pd

## API를 통한 소환사 정보 수집

In [209]:
# Lol Api에서 소환사 명을 통해서 게임 정보를 알 수 있는 puuid 불러오기 
summoner_Name = 'one summer drive' #소환사명
api_key = "RGAPI-fa3757e6-edbf-4e39-8001-14e03ad13b01" # api key
request_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/103.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Charset": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://developer.riotgames.com",
    "X-Riot-Token": "RGAPI-fa3757e6-edbf-4e39-8001-14e03ad13b01"
}

summoner_url="https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key

In [210]:
# requests를 통해서 puuid 추출하기
def summoner(summoner_Name, api_key):
    summoner_url = "https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key
    return requests.get(summoner_url, headers=request_headers).json()['puuid']
summoner_puuid = summoner(summoner_Name, api_key)
summoner_puuid

'cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72J0XfluBWZsNURa-_rKA5AR_vMndVW8tg'

In [211]:
# match id를 확인할 수 있는 api를 통해서 조회하기
# match_count : 조회하고 싶은 매치의 수
# 더 많이 조회를 하고 싶으나 횟수 제한이 있다. (100) 이번 시즌 내가 플레이한 62게임에 대해서 조사하기 
def match(summoner_puuid, match_count):
    match_url = "https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/"+summoner_puuid+"/ids?start=0&count="+str(match_count)
    return requests.get(match_url, headers=request_headers).json()
match_id = match(summoner_puuid, 90)
match_id

['KR_6017207879',
 'KR_6017033024',
 'KR_6016828310',
 'KR_6016772762',
 'KR_6016770544',
 'KR_6016651859',
 'KR_6016575318',
 'KR_6015787618',
 'KR_6015488822',
 'KR_6015088138',
 'KR_6014287649',
 'KR_6012864614',
 'KR_6012476838',
 'KR_6011712595',
 'KR_6011597486',
 'KR_6010856241',
 'KR_6009908963',
 'KR_6009876008',
 'KR_6009892943',
 'KR_6009828517',
 'KR_6009775487',
 'KR_6009753833',
 'KR_6008388001',
 'KR_6008101825',
 'KR_6007718428',
 'KR_6007632404',
 'KR_6007543140',
 'KR_6007407898',
 'KR_6007030534',
 'KR_6006751091',
 'KR_6006310180',
 'KR_6006082114',
 'KR_6005589922',
 'KR_6005521617',
 'KR_6005292412',
 'KR_6004906534',
 'KR_6002949666',
 'KR_6002335329',
 'KR_6000663825',
 'KR_6000095173',
 'KR_5995456456',
 'KR_5993880535',
 'KR_5992325428',
 'KR_5992263876',
 'KR_5992247758',
 'KR_5992174929',
 'KR_5992139509',
 'KR_5992067375',
 'KR_5990749357',
 'KR_5990208811',
 'KR_5989449386',
 'KR_5984608595',
 'KR_5983799498',
 'KR_5975922818',
 'KR_5972653802',
 'KR_59718

In [212]:
# 게임 세부내용 불러오기
# 조회는 하나씩만 가능하므로 for문이나 while문을 돌려 하나씩 조회한다.
def game(match_id_num):
    game_url = "https://asia.api.riotgames.com/lol/match/v5/matches/"+match_id_num
    return requests.get(game_url, headers=request_headers).json()['info']['participants']


## 분석에 사용할 90개의 데이터 수집하여 데이터 프레임 생성
- 100개로 하고 돌렸으나 API응답문제가 발생하였다

In [216]:
#Future Warning 메세지 제거하기
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# 게임 데이터를 담을 DataFrame 생성하기
game_df = pd.DataFrame(columns=['puuid', 'teamPosition', 'individualPosition', 'champName', 'champExp', 'summoner_spell1', 'summoner_spell2' ,
                                'kills', 'deaths', 'assists', 'damageDealtToBuildings', 'totalDamageDealtToChamp', 'damagePerMin', 'teamDamagePer(%)',
                                'killParticipation(%)', 'totalDamageTaken', 'damageTakeOnTeamPer(%)', 'goldEarned', 'goldPerMin', 'totalCs',
                                'maxCsAdvantageOnLaneOpponent', 'maxLevelLeadLaneOpponent', 'gameEndedInEarlySurren', 'gameEndedInSurren',
                                'teamEarlySurrn', 'timePlayed'])

for i in range(len(match_id)):
    print(i)
    game_content = game(match_id[i])
    for j in range(len(game_content)):
        try:
            input_data = {

                'puuid':game_content[j]['puuid'], 
                'teamPosition':game_content[j]['teamPosition'], 
                'individualPosition':game_content[j]['individualPosition'], 
                'champName':game_content[j]['championName'], 
                'champExp':float(game_content[j]['champExperience']), 
                'summoner_spell1':float(game_content[j]['summoner1Id']), 
                'summoner_spell2':float(game_content[j]['summoner2Id']),
                'kills':float(game_content[j]['kills']), 
                'deaths':float(game_content[j]['deaths']), 
                'assists':float(game_content[j]['assists']), 
                'damageDealtToBuildings':float(game_content[j]['damageDealtToBuildings']), 
                'totalDamageDealtToChamp':float(game_content[j]['totalDamageDealtToChampions']),
                'damagePerMin':game_content[j]['challenges']['damagePerMinute'], 
                'teamDamagePer(%)':round(game_content[j]['challenges']['teamDamagePercentage'], 2)*100,
                'killParticipation(%)':game_content[j]['challenges']['killParticipation']*100,
                'totalDamageTaken':float(game_content[j]['totalDamageTaken']),
                'damageTakeOnTeamPer(%)':round(game_content[j]['challenges']['damageTakenOnTeamPercentage'],2)*100,
                'goldEarned':float(game_content[j]['goldEarned']),
                'goldPerMin':game_content[j]['challenges']['goldPerMinute'],
                'totalCs':float(game_content[j]['totalMinionsKilled']+game_content[j]['neutralMinionsKilled']),
                'maxCsAdvantageOnLaneOpponent':float(game_content[j]['challenges']['maxCsAdvantageOnLaneOpponent']),
                'maxLevelLeadLaneOpponent':float(game_content[j]['challenges']['maxLevelLeadLaneOpponent']),
                'gameEndedInEarlySurren':game_content[j]['gameEndedInEarlySurrender'],
                'gameEndedInSurren':game_content[j]['gameEndedInSurrender'],
                'teamEarlySurrn':game_content[j]['teamEarlySurrendered'],
                'timePlayed':float(round(game_content[j]['timePlayed']/60,2)*100) # 4자리로 앞의 2자리는 분, 뒤의 2자리는 초로 구성되어있다. 
                }
            game_df = game_df.append(input_data, ignore_index=True)
        except:
            input_data = {

                'puuid':game_content[j]['puuid'], 
                'teamPosition':game_content[j]['teamPosition'], 
                'individualPosition':game_content[j]['individualPosition'], 
                'champName':game_content[j]['championName'], 
                'champExp':float(game_content[j]['champExperience']), 
                'summoner_spell1':float(game_content[j]['summoner1Id']), 
                'summoner_spell2':float(game_content[j]['summoner2Id']),
                'kills':float(game_content[j]['kills']), 
                'deaths':float(game_content[j]['deaths']), 
                'assists':float(game_content[j]['assists']), 
                'damageDealtToBuildings':float(game_content[j]['damageDealtToBuildings']), 
                'totalDamageDealtToChamp':float(game_content[j]['totalDamageDealtToChampions']),
                'damagePerMin':game_content[j]['challenges']['damagePerMinute'], 
                'teamDamagePer(%)':round(game_content[j]['challenges']['teamDamagePercentage'], 2)*100,
                'killParticipation(%)':game_content[j]['challenges']['killParticipation']*100,
                'totalDamageTaken':float(game_content[j]['totalDamageTaken']),
                'damageTakeOnTeamPer(%)':round(game_content[j]['challenges']['damageTakenOnTeamPercentage'],2)*100,
                'goldEarned':float(game_content[j]['goldEarned']),
                'goldPerMin':game_content[j]['challenges']['goldPerMinute'],
                'totalCs':float(game_content[j]['totalMinionsKilled']+game_content[j]['neutralMinionsKilled']),
                'maxCsAdvantageOnLaneOpponent':0,
                'maxLevelLeadLaneOpponent':0,
                'gameEndedInEarlySurren':game_content[j]['gameEndedInEarlySurrender'],
                'gameEndedInSurren':game_content[j]['gameEndedInSurrender'],
                'teamEarlySurrn':game_content[j]['teamEarlySurrendered'],
                'timePlayed':float(round(game_content[j]['timePlayed']/60,2)*100) # 4자리로 앞의 2자리는 분, 뒤의 2자리는 초로 구성되어있다. 
                }
            game_df = game_df.append(input_data, ignore_index=True) 

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89


In [217]:
game_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   puuid                         900 non-null    object 
 1   teamPosition                  900 non-null    object 
 2   individualPosition            900 non-null    object 
 3   champName                     900 non-null    object 
 4   champExp                      900 non-null    float64
 5   summoner_spell1               900 non-null    float64
 6   summoner_spell2               900 non-null    float64
 7   kills                         900 non-null    float64
 8   deaths                        900 non-null    float64
 9   assists                       900 non-null    float64
 10  damageDealtToBuildings        900 non-null    float64
 11  totalDamageDealtToChamp       900 non-null    float64
 12  damagePerMin                  900 non-null    float64
 13  teamD

In [218]:
game_df

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Akali,17035.0,4.0,12.0,5.0,10.0,4.0,...,19.0,11279.0,292.662528,167.0,3.00,0.0,False,False,False,3853.0
1,ef8EdZNmTuHH49lNib4mUFIWu-a60p8EvU-K1e9dzXXHQn...,JUNGLE,JUNGLE,Taliyah,17337.0,11.0,4.0,3.0,11.0,7.0,...,21.0,13305.0,345.221804,219.0,40.30,1.0,False,False,False,3853.0
2,4QnucqF7Qx3g6qdDWKZm8naW8QU2rIU11xPOcMRq5XbjFm...,MIDDLE,MIDDLE,Yone,19124.0,14.0,4.0,9.0,10.0,8.0,...,20.0,16345.0,424.098371,237.0,96.00,1.0,False,False,False,3853.0
3,aOAvT9veq32BJqJZaAv4RU9SZS5pkeNXjZKT9VzQD9j8m0...,BOTTOM,BOTTOM,Kalista,17249.0,7.0,4.0,13.0,9.0,8.0,...,25.0,18717.0,485.660739,284.0,62.80,2.0,False,False,False,3853.0
4,eqzWrrEr4D_zWSjTyf3VDSvffdonhAUGA2VqTHu46QD3YO...,UTILITY,UTILITY,MissFortune,14933.0,4.0,3.0,3.0,9.0,15.0,...,15.0,12123.0,314.549904,58.0,26.40,3.0,False,False,False,3853.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,YYD-vZSBRgTjGuiw6PFuQBkq_jWg4Gs1qd3QMNzxfpNnBS...,TOP,TOP,Nasus,15778.0,4.0,12.0,6.0,4.0,4.0,...,29.0,12352.0,399.182742,192.0,35.05,2.0,False,False,False,3093.0
896,2Oz57YFmjbhaQ901AdAur8KCg_Yrihi6Q43hA66isY2Zdi...,JUNGLE,JUNGLE,Vi,13183.0,11.0,4.0,7.0,11.0,9.0,...,19.0,11292.0,364.925387,115.0,3.00,1.0,False,False,False,3093.0
897,pEl9IfL_Dvj7AVHiYUxXGONzTJkYe02DkHRhx7soMjA3Tt...,MIDDLE,MIDDLE,Yasuo,9266.0,14.0,4.0,0.0,15.0,2.0,...,18.0,7362.0,237.938166,106.0,1.00,0.0,False,False,False,3093.0
898,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,12242.0,4.0,7.0,7.0,6.0,7.0,...,14.0,13746.0,444.223697,189.0,17.00,1.0,False,False,False,3093.0


## 가설 확인하기

```
인게임 속의 트롤링은 처음부터 발생할 수도 있고 중간에 소환사의 변심으로 인해서 발생할 수도 있고 수 많은 경우로 인해 발생한다.
기본적으로 의도적인 죽음, 다른 소환사들 방해, 한타에 합류를 하지 않거나, 우물에서 잠수를 하거나 
위와 같은 행위들은 기본적으로 DPM의 하락을 가져온다. 또한 안정적으로 성장을 한 상대 라이너에 비해서 성장 측면에서 많은 차이가 나타난다. 

픽창에서의 싸움으로 인한 소환사 스펠, 주 포지션이 아니므로 인한 트롤 등에 대해서는 소환사 스펠을 통해서 확인한다. 
```

- 가장 최근 게임의 트롤은 워익이었다. 
- 욕설과 함께 한타참여X, 정글링만 하거나, 우물에서 잠수 
- 딜량에 대해서 수치화 하기 
- 게임은 빠른 서렌이 나오는 15분부터 끝나기 시작한다.
- 15분까지는 기본적으로 라인전이 끝나는 시기, 탑, 미드, 바텀의 경우에는 라인전을 통해서 딜량을 쌓을수 있으나 정글의 경우에는 힘들다. 
- 게임이 장기간으로 가는경우 서포터는 딜량이 작기 마련이다. 
- 원딜의 경우에는 코어가 뜨면 뜰 수록 강해진다. 
- 위와 같은 경우를 수치화 조정해서 딜량을 계산한다.

## Match에 포함되어 있는 칼바람, 일반게임 정보 제거하기 

In [136]:
# 포지션에 Top, Jungle, Middle, Botoom, Utility 이외의 공백인 행들이 일반게임, 칼바람나락의 결과
game_df['teamPosition'].unique()

array(['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY', ''], dtype=object)

In [163]:
game_df['teamPosition']==''.index

0      False
1      False
2      False
3      False
4      False
       ...  
895    False
896    False
897    False
898    False
899    False
Name: teamPosition, Length: 900, dtype: bool

In [169]:
# 포지션 선택이 없는 게임 제거하기
# 29게임이 제외되었다. 
game_df = game_df.drop(game_df[game_df['teamPosition']==''].index, axis=0).reset_index(drop=True)
game_df

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timeplayed,timePlayed
0,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Akali,17035,4,12,5,10,4,...,11279,292.662528,167,3,0,False,False,False,NaN,3853.0
1,ef8EdZNmTuHH49lNib4mUFIWu-a60p8EvU-K1e9dzXXHQn...,JUNGLE,JUNGLE,Taliyah,17337,11,4,3,11,7,...,13305,345.221804,219,40.3,1,False,False,False,NaN,3853.0
2,4QnucqF7Qx3g6qdDWKZm8naW8QU2rIU11xPOcMRq5XbjFm...,MIDDLE,MIDDLE,Yone,19124,14,4,9,10,8,...,16345,424.098371,237,96.0,1,False,False,False,NaN,3853.0
3,aOAvT9veq32BJqJZaAv4RU9SZS5pkeNXjZKT9VzQD9j8m0...,BOTTOM,BOTTOM,Kalista,17249,7,4,13,9,8,...,18717,485.660739,284,62.8,2,False,False,False,NaN,3853.0
4,eqzWrrEr4D_zWSjTyf3VDSvffdonhAUGA2VqTHu46QD3YO...,UTILITY,UTILITY,MissFortune,14933,4,3,3,9,15,...,12123,314.549904,58,26.4,3,False,False,False,NaN,3853.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605,YYD-vZSBRgTjGuiw6PFuQBkq_jWg4Gs1qd3QMNzxfpNnBS...,TOP,TOP,Nasus,15778,4,12,6,4,4,...,12352,399.182742,192,35.05,2,False,False,False,NaN,3093.0
606,2Oz57YFmjbhaQ901AdAur8KCg_Yrihi6Q43hA66isY2Zdi...,JUNGLE,JUNGLE,Vi,13183,11,4,7,11,9,...,11292,364.925387,115,3,1,False,False,False,NaN,3093.0
607,pEl9IfL_Dvj7AVHiYUxXGONzTJkYe02DkHRhx7soMjA3Tt...,MIDDLE,MIDDLE,Yasuo,9266,14,4,0,15,2,...,7362,237.938166,106,1,0,False,False,False,NaN,3093.0
608,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,12242,4,7,7,6,7,...,13746,444.223697,189,17,1,False,False,False,NaN,3093.0


### Play시간대 별로 나누기 

#### 20분 이내 종료된 게임

In [170]:
# 5게임만이 61게임중에서 빠르게 끝난 게임 
game_played20down = game_df[game_df['timePlayed']<2000].reset_index(drop=True)
game_played20down

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timeplayed,timePlayed
0,9yf8Rs_z-wYgmGrOEt3c5Dxt5vxXciWjSutOB3cJe_X5_b...,TOP,TOP,Jax,6558,12,4,2,7,0,...,5036,320.051088,80,1,0,False,True,False,NaN,1573.0
1,VUCm_CvzaECN6ltgCfgnj4tPkV98sMcR0qOK-Fe72GKuRm...,JUNGLE,JUNGLE,RekSai,5310,11,4,2,2,2,...,4776,303.528755,70,0,0,False,True,False,NaN,1573.0
2,CyTdqikqKhul9jBHBoUm08M8f29sDoPbnjfSeBq1bdP3h_...,MIDDLE,MIDDLE,Vex,7538,4,14,4,2,3,...,6589,418.764576,91,41,4,False,True,False,NaN,1573.0
3,861oCjkJqJ0Px5b1qomocugo1AVxsEQCO125KUAe2_ks8x...,BOTTOM,BOTTOM,MissFortune,5804,7,4,3,0,2,...,6060,385.147985,114,7,1,False,True,False,NaN,1573.0
4,0DZwozk2_QL_6JM7JdXg_P2DDtIRsiazbcb2KRGZdrJbn-...,UTILITY,UTILITY,Blitzcrank,5272,4,3,1,1,4,...,4161,264.481713,22,10,1,False,True,False,NaN,1573.0
5,qgPQotn5woPPNGVWHTMKkNsxSO15eUKgKArQWn2D06bxpW...,TOP,TOP,Darius,8809,4,6,8,2,0,...,7298,463.812662,117,45,3,False,True,False,NaN,1573.0
6,c3PQr4ImXyUqbkgaZzE-V8V-MnByQjA37QG5o7LfQknS5B...,JUNGLE,JUNGLE,Kayn,7686,11,4,2,2,3,...,6618,420.571086,118,50.0,2,False,True,False,NaN,1573.0
7,IeIOTtJf8Oisc_iGCJWABPZXIpCmA9bNNYKIIIyI_-a5R6...,MIDDLE,MIDDLE,Katarina,3969,14,12,2,4,1,...,4159,264.319990,50,8,1,False,True,False,NaN,1573.0
8,tg4UHF9QI9Z3SgiNekYIiyf2c9rHc2IKBdwYlaHdAvajTE...,BOTTOM,BOTTOM,Ezreal,4795,7,4,0,2,0,...,4611,293.043398,111,21,1,False,True,False,NaN,1573.0
9,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,UTILITY,UTILITY,Ashe,4859,4,3,0,2,0,...,3449,219.203407,30,8,1,False,True,False,NaN,1573.0


#### 20분 ~ 30분 사이 종료된 게임

In [183]:
# 20분에서 30분 사이 종료된 게임으로는 33게임이 존재한다. 
game_played2030 = game_df[(2000<= game_df['timePlayed']) & (game_df['timePlayed'] < 3000)].reset_index(drop=True)
game_played2030

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timeplayed,timePlayed
0,r1mEdSqYA-GlBBYPlXV_uq4ACKssZg0SHp1u0H56snJMsB...,TOP,TOP,Mordekaiser,11967,6,4,5,1,0,...,7839,386.875311,150,39,2,False,True,False,NaN,2022.0
1,1gA6H322Xs_UU0AXiX9VNoe3OjZqj-EmFUAkvqfkVj5cXt...,JUNGLE,JUNGLE,Vi,6535,11,4,1,1,3,...,6161,304.100253,93,6.5,1,False,True,False,NaN,2022.0
2,2W_zqCNWa1-3zcBCzzoTXjZg_x1lbmV4224tpSRDQh5dH9...,MIDDLE,MIDDLE,Corki,8902,12,4,3,0,1,...,6974,344.227771,140,42,1,False,True,False,NaN,2022.0
3,6FZHxzDQzUXFSTaNXHdJYhiPRGMh1dC74Uns4RbgtLqhg-...,BOTTOM,BOTTOM,Sivir,8561,4,7,4,0,6,...,8588,423.862195,158,43,2,False,True,False,NaN,2022.0
4,A7p3TM0ZlmOWclAHCxcUXw1tyd13cy8-EJ4vkCMFNEu0cz...,UTILITY,UTILITY,Xerath,6240,14,4,4,0,4,...,7070,348.953532,19,9,2,False,True,False,NaN,2022.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,QxffUyKLIHztu5DPi06OglE9Sf8Gr2NdsNY1d7EhFRfJpe...,TOP,TOP,LeeSin,7324,12,4,3,8,3,...,6656,317.423426,80,17,1,False,True,False,NaN,2097.0
326,7laSMnqFaZQgu3zqX_GVtjihk_AykGaA-0xIwjOB_20C1G...,JUNGLE,JUNGLE,Zac,5613,11,4,1,9,2,...,5155,245.845942,66,5,1,False,True,False,NaN,2097.0
327,MmWFm-YgIIxp0naNdqhU4hMmW00aI9V8dVM5mCQDdXsipx...,MIDDLE,MIDDLE,Yasuo,9404,14,4,6,6,4,...,10991,524.114813,151,65.0,1,False,True,False,NaN,2097.0
328,tPblUi0-8C2VmSudF5TeoPjvU2ifuXGZfYDHQHGDiirJ7L...,BOTTOM,UTILITY,Jhin,6091,7,4,1,8,3,...,5339,254.619547,70,40,1,False,True,False,NaN,2097.0


#### 30분 ~ 40분 사이 종료된 게임

In [185]:
# 20게임이 존재한다.
game_played3040 = game_df[(3000<= game_df['timePlayed']) & (game_df['timePlayed'] < 4000)].reset_index(drop=True)
game_played3040

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timeplayed,timePlayed
0,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Akali,17035,4,12,5,10,4,...,11279,292.662528,167,3,0,False,False,False,NaN,3853.0
1,ef8EdZNmTuHH49lNib4mUFIWu-a60p8EvU-K1e9dzXXHQn...,JUNGLE,JUNGLE,Taliyah,17337,11,4,3,11,7,...,13305,345.221804,219,40.3,1,False,False,False,NaN,3853.0
2,4QnucqF7Qx3g6qdDWKZm8naW8QU2rIU11xPOcMRq5XbjFm...,MIDDLE,MIDDLE,Yone,19124,14,4,9,10,8,...,16345,424.098371,237,96.0,1,False,False,False,NaN,3853.0
3,aOAvT9veq32BJqJZaAv4RU9SZS5pkeNXjZKT9VzQD9j8m0...,BOTTOM,BOTTOM,Kalista,17249,7,4,13,9,8,...,18717,485.660739,284,62.8,2,False,False,False,NaN,3853.0
4,eqzWrrEr4D_zWSjTyf3VDSvffdonhAUGA2VqTHu46QD3YO...,UTILITY,UTILITY,MissFortune,14933,4,3,3,9,15,...,12123,314.549904,58,26.4,3,False,False,False,NaN,3853.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,YYD-vZSBRgTjGuiw6PFuQBkq_jWg4Gs1qd3QMNzxfpNnBS...,TOP,TOP,Nasus,15778,4,12,6,4,4,...,12352,399.182742,192,35.05,2,False,False,False,NaN,3093.0
196,2Oz57YFmjbhaQ901AdAur8KCg_Yrihi6Q43hA66isY2Zdi...,JUNGLE,JUNGLE,Vi,13183,11,4,7,11,9,...,11292,364.925387,115,3,1,False,False,False,NaN,3093.0
197,pEl9IfL_Dvj7AVHiYUxXGONzTJkYe02DkHRhx7soMjA3Tt...,MIDDLE,MIDDLE,Yasuo,9266,14,4,0,15,2,...,7362,237.938166,106,1,0,False,False,False,NaN,3093.0
198,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,12242,4,7,7,6,7,...,13746,444.223697,189,17,1,False,False,False,NaN,3093.0


#### 40분 이후에 종료된 게임

In [187]:
# 3게임이 존재한다. 
game_played40up = game_df[4000 < game_df['timePlayed']].reset_index(drop=True)
game_played40up

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timeplayed,timePlayed
0,IlqYon3McqGJG0th5WG1bYTFyUPDUIMlScWszvvMqQt7Qd...,TOP,TOP,Poppy,25408,12,4,8,5,7,...,18877,391.931189,307,77,3,False,True,False,NaN,4815.0
1,1Ki4PlEmbDX2h5__VdSYZA5tXea8oJbbbq3mLcbJWTjVGa...,JUNGLE,JUNGLE,Janna,19322,11,4,1,15,22,...,12124,251.734976,98,4,1,False,True,False,NaN,4815.0
2,3twz4G44Wj1kWZMPLSrDeGxCVp8ARFprdFiEAPxNK6o0zk...,MIDDLE,MIDDLE,Akali,22563,14,4,26,9,2,...,22486,466.869134,235,26,1,False,True,False,NaN,4815.0
3,FUwkQVEcZbRUb7d_uiY9nBHK4QPca6LBO1Io4DH9Ua2icm...,BOTTOM,BOTTOM,Jinx,24119,7,4,8,8,16,...,19669,408.379721,370,150.0,2,False,True,False,NaN,4815.0
4,ihATf3I8tbukfaBkTGtNcQfQ2_0XzVGOSIWojWwTsjUHDL...,UTILITY,UTILITY,Xerath,20470,14,4,9,6,11,...,16612,344.908673,103,40,2,False,True,False,NaN,4815.0
5,Gnj1h1FYYZIqGMaPwKoetO_jbKnUvUvukKaNXUsz9uTHV8...,TOP,TOP,Jax,21418,14,4,7,13,6,...,16747,347.702385,253,20,1,False,True,False,NaN,4815.0
6,Dfl5W0HF1Y93K-sRuIeVV6pH3r7eq8zqIVB2UJ63N8CDrf...,JUNGLE,JUNGLE,Viego,20617,11,4,9,7,15,...,17344,360.108521,186,89.0,3,False,True,False,NaN,4815.0
7,bvZq5jVAvK8_TY9gKPGSKx2mXzFBVZqGfrs1IuqWvMcLZt...,MIDDLE,MIDDLE,Ziggs,23850,7,4,6,10,16,...,19079,396.125894,328,96.5,2,False,True,False,NaN,4815.0
8,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,18809,4,7,18,9,6,...,20659,428.942018,220,12,2,False,True,False,NaN,4815.0
9,8SIB8WFW_M_an1ElcXOEkw4u4gMcURUMkSusKBw12l4AUb...,UTILITY,UTILITY,Senna,19423,14,4,3,13,21,...,13826,287.058017,73,6,2,False,True,False,NaN,4815.0


### 종료된 시간 대별, 라인별 평균 딜량, 챔피언 경험치, 골드획득을 확인한다. 

In [ ]:
def gamemean(time_df):
    

In [189]:
game_played40up.groupby('teamPosition').mean()

,damagePerMin,teamDamagePer(%),killParticipation(%),damageTakeOnTeamPer(%),goldPerMin,timePlayed
teamPosition,,,,,,
BOTTOM,758.532973,20.333333,50.536066,16.333333,415.969559,4466.666667
JUNGLE,678.084230,17.833333,47.381286,27.000000,383.181858,4466.666667
MIDDLE,843.231130,23.166667,45.404536,19.833333,410.206902,4466.666667
TOP,767.476149,20.666667,36.256642,23.500000,394.521466,4466.666667
UTILITY,699.504360,18.166667,47.708037,13.833333,304.895884,4466.666667
